# 3.1 Modality specificity

Configure the analysis, load saved native weights and coverage, inspect weight magnitudes, and compare spatial cosine profiles and normalized semivariance. Gradient and weight-clustering analyses follow. Expensive computations run through the printed terminal commands; plots use saved results. Previous code is archived in `old/modality_specificity_before_restructure/`.


In [ ]:
from pathlib import Path
import sys
import json
import shlex
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from nilearn import plotting

ROOT = Path.cwd()
MEG_KIND = 'paired_coverage'
MODEL = 'plssvd'  # separate_pca, plssvd, joint_pca
BLOCK_SCALING = True
N_COMPONENTS = 10
CONDITION_MODE = 'average'
MEG_SCALING = 'channel_zscore'  # 'none' disables the correction
IEEG_MULTIPLIER = 1000.  # 1. leaves amplitudes unchanged
SEED = 2026
MAX_GRAM_GIB = 2.

for path in (ROOT, ROOT.parent, ROOT.parent / 'LB'):
    sys.path.insert(0, str(path))
from vs1_coverage_figures import MODALITY_COLORS, apply_style
from vs3_1_run_modality_specificity import prepare_analysis, result_directory
apply_style()
plt.rcParams['axes.grid'] = True

# Repeated associations use the fixed half-trial subset from PAIRING_ONLY.
N_PAIRING_ITERATIONS = 100
PAIRING_SPLIT_UNIT = 'trial'
PAIRING_CACHE_DIR = ROOT / 'out' / 'trial_cache'
PAIRING_EXISTING_DIR = ROOT / 'out' / 'weight_stability' / 'paired_coverage' / 'pairing'
PAIRING_ATLAS_IMAGE = None  # Optional atlas for newly computed fits
PAIRING_ATLAS_LABELS = None


## Load fitted weights and metadata

Set input paths and preprocessing corrections here. `BLOCK_SCALING=True` gives each constructed modality equal total variance at model fitting. Random subject associations use `SEED`. Results are stored separately for each configuration. `RECOMPUTE=True` prints a recomputation command; it does not launch a job. After the job completes, set it back to `False` and reload. Recompute after changing input files at unchanged paths.


In [ ]:
INPUT_DIR = ROOT
OUTPUT_DIR = ROOT / 'out' / 'modality_specificity' / MEG_KIND / MODEL
RECOMPUTE = False
MEG_DIR = INPUT_DIR / 'MEG' / 'dataMEG'
IEEG_DIR = INPUT_DIR / 'ieeg_shortWOBS_fs250'
METADATA_CSV = None
MEG_COORDINATE_UNIT = 'm'
SFREQ = 250.
CONDITIONS = [1, 2]

config = dict(output_dir=str(OUTPUT_DIR), fit=dict(
    model=MODEL, meg_kind=MEG_KIND, n_components=N_COMPONENTS,
    block_scaling=BLOCK_SCALING, condition_mode=CONDITION_MODE,
    seed=SEED, max_gram_gib=MAX_GRAM_GIB,
    data=dict(meg_dir=str(MEG_DIR), ieeg_dir=str(IEEG_DIR),
        metadata_csv=str(METADATA_CSV) if METADATA_CSV else None,
        ieeg_coordinate_unit='mm', meg_coordinate_unit=MEG_COORDINATE_UNIT,
        sfreq=SFREQ, conditions=CONDITIONS, meg_scaling=MEG_SCALING,
        ieeg_scaling='none', ieeg_multiplier=IEEG_MULTIPLIER)), analyses={})
base = prepare_analysis(config, 'weights', OUTPUT_DIR / 'weights_config.json', RECOMPUTE)
weights = coverage_region_metadata = region_count_table = None
if base is not None:
    weights, coverage_region_metadata, region_count_table = base['weights'], base['metadata'], base['counts']
    display(pd.DataFrame([dict(modality=m, features=len(frame),
        subject_labels=frame[next(k for k in ('meg_subject', 'subject', 'ieeg_subject') if k in frame)].nunique(), components=weights[m].shape[1])
        for m, frame in coverage_region_metadata.items()]))


## Regional coverage: electrode/source counts

Counts describe native feature rows, including co-located measurements. Averaged MEG rows are sources, not original participant recordings.


In [ ]:
FIGSIZE = (16, 6)
if base is not None:
    regions = region_count_table.region.drop_duplicates().tolist()
    display(region_count_table.pivot(index='region', columns='modality', values='n_channels').reindex(regions))
    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, layout='constrained')
    for ax, modality in zip(axes, ['iEEG', 'MEG']):
        values = region_count_table.query('modality == @modality').set_index('region').n_channels.reindex(regions)
        ax.bar(regions, values, color=MODALITY_COLORS[modality])
        ax.set(title=modality, ylabel='Number of electrodes / sources')
        ax.tick_params(axis='x', rotation=90, labelsize=8)
        ax.grid(axis='y', alpha=.2)
    plt.show()


## Magnitudes and component-specific thresholds across associations

Reuse compatible `PAIRING_ONLY` fits or compute new fits using the same fixed half-trial subset and random streams. This section uses **paired-coverage MEG**, independently of `MEG_KIND` used for the native-weight analyses below. Corrections must match the stability workflow: iEEG ×1000 and MEG z-scoring from training condition means.

Components are matched to iteration 0 using training time courses; coupled models retain their shared cross-modal axes. Standardization (when enabled) and the magnitude percentile are calculated separately for each aligned component and association. Thresholded weights include zeroed locations. Histograms and participant-balanced ROI means show mean ± sample SD across associations. Brain maps show the mean thresholded magnitude at **fixed iEEG electrode anchors**, including for MEG; these are not native MEG source maps.

Thresholds, component correspondence, per-association histograms and ROI means are saved as CSV tables beside the result file. SD describes association sensitivity, not a confidence interval.


In [ ]:
INPUT_DIR = PAIRING_EXISTING_DIR
OUTPUT_DIR = Path(config['output_dir'])
RECOMPUTE = False
NORMALIZE_WEIGHTS = True
PERCENTILE = 95
HISTOGRAM_BINS = 50

config['pairing'] = dict(existing_dir=str(INPUT_DIR), cache_dir=str(PAIRING_CACHE_DIR),
    iterations=N_PAIRING_ITERATIONS, split_unit=PAIRING_SPLIT_UNIT,
    atlas_image=str(PAIRING_ATLAS_IMAGE) if PAIRING_ATLAS_IMAGE else None,
    atlas_labels=str(PAIRING_ATLAS_LABELS) if PAIRING_ATLAS_LABELS else None)
config['analyses']['pairing_weights'] = dict(normalize=NORMALIZE_WEIGHTS,
    percentile=PERCENTILE, bins=HISTOGRAM_BINS)
pairing_weights = prepare_analysis(config, 'pairing_weights',
    OUTPUT_DIR / 'pairing_weights_config.json', RECOMPUTE)
if pairing_weights is not None:
    print('Source fits:', pairing_weights['source_dir'])
    print('Threshold CSV:', result_directory(config, 'pairing_weights') / 'thresholds.csv')


In [ ]:
OUTPUT_DIR = result_directory(config, 'pairing_weights') / 'figures'
COMPONENTS_TO_PLOT = min(3, N_COMPONENTS)
FIGSIZE = (18, 15)

if pairing_weights is not None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    anchor = pairing_weights['anchor']
    coordinates = anchor[['x', 'y', 'z']].to_numpy()
    roi_order = sorted(pairing_weights['roi_iterations'].region.unique())
    for component in range(1, COMPONENTS_TO_PLOT + 1):
        fig, axes = plt.subplots(3, 2, figsize=FIGSIZE, layout='constrained')
        for column, modality in enumerate(['iEEG', 'MEG']):
            color = MODALITY_COLORS[modality]
            hist = pairing_weights['histogram_iterations'].query('modality == @modality and component == @component')
            hist = hist.groupby('magnitude')['count'].agg(['mean', 'std'])
            axes[0, column].plot(hist.index, hist['mean'], color=color)
            axes[0, column].fill_between(hist.index, hist['mean']-hist['std'], hist['mean']+hist['std'], color=color, alpha=.2)
            thr = pairing_weights['thresholds'].query('modality == @modality and component == @component').threshold
            axes[0, column].axvline(thr.mean(), color='crimson', linestyle='--', label='Mean threshold')
            axes[0, column].axvspan(thr.mean()-thr.std(), thr.mean()+thr.std(), color='crimson', alpha=.1)
            axes[0, column].set(title=f'{modality} · component {component}',
                xlabel='Absolute standardized weight' if NORMALIZE_WEIGHTS else 'Absolute weight',
                ylabel='Feature count: mean ± SD across associations')
            axes[0, column].legend()
            magnitude = pairing_weights['maps'][modality]['mean'][:, component-1]
            selected = magnitude > 0
            if selected.any():
                plotting.plot_markers(magnitude[selected], coordinates[selected], figure=fig, axes=axes[1, column],
                    node_cmap='Reds', node_vmin=0, node_size=2, display_mode='ortho',
                    title=f'{modality}: mean thresholded magnitude at iEEG anchors')
            else:
                plotting.plot_glass_brain(None, figure=fig, axes=axes[1, column], title='No retained magnitude')
            roi = pairing_weights['roi_iterations'].query('modality == @modality and component == @component')
            roi = roi.groupby('region').magnitude.agg(['mean','std']).reindex(roi_order)
            axes[2, column].bar(np.arange(len(roi)), roi['mean'], yerr=roi['std'], capsize=3, color=color, alpha=.8)
            axes[2, column].set(xticks=np.arange(len(roi)), xticklabels=roi.index,
                ylabel='Thresholded magnitude: mean ± association SD', title='Equal participant contribution within ROI')
            axes[2, column].tick_params(axis='x', rotation=90, labelsize=8)
            axes[0, column].grid(alpha=.2)
            axes[2, column].grid(axis='y', alpha=.2)
        fig.suptitle(f'{MODEL} · paired coverage · {N_PAIRING_ITERATIONS} fixed-trial associations')
        fig.savefig(OUTPUT_DIR / f'pairing_weights_component_{component}.png', dpi=180, bbox_inches='tight')
        plt.show()


In [ ]:
FIGSIZE = (12, 3 * min(3, N_COMPONENTS))
THRESHOLD_COMPONENTS = list(range(1, min(3, N_COMPONENTS) + 1))

if pairing_weights is not None:
    fig, axes = plt.subplots(len(THRESHOLD_COMPONENTS), 1, figsize=FIGSIZE, squeeze=False, layout='constrained')
    for ax, component in zip(axes.flat, THRESHOLD_COMPONENTS):
        for modality, color in MODALITY_COLORS.items():
            table = pairing_weights['thresholds'].query('modality == @modality and component == @component').sort_values('iteration')
            ax.plot(table.iteration + 1, table.threshold, 'o-', color=color, label=modality, markersize=3)
        ax.set(title=f'Component {component} · percentile {PERCENTILE}', xlabel='Association iteration', ylabel='Magnitude threshold')
        ax.grid(alpha=.2)
        ax.legend()
    plt.show()


## Spatial cosine profiles and normalized semivariance across associations

Use the first `PROFILE_N_COMPONENTS` **aligned** axes from the saved fits. Absolute-weight cosine similarity describes component proportions; normalized semivariance describes weight-norm differences. Matching exclusions and distance bins are evaluated separately for each association using native coordinates; only brain summaries use fixed electrode anchors.

Each participant pair contributes one curve per distance bin. Panel 1 shows the mean of association-level means with sample SD across associations. Panel 2 shows the mean and sample SD across participant pairs **within `PROFILE_ASSOCIATION`**, selected below (one-based). These sources of variation are distinct and neither band is a confidence interval. The third cosine panel retains excess similarity above spatial shuffling, now summarized across associations. Dashed lines show corresponding spatial-shuffle means. Undefined bins remain missing; saved tables report contributing participant and association counts.


In [ ]:
INPUT_DIR = result_directory(config, 'pairing_weights')
OUTPUT_DIR = Path(config['output_dir'])
RECOMPUTE = False
PROFILE_N_COMPONENTS = min(2, N_COMPONENTS)
PROFILE_ASSOCIATION = 1  # One-based association for within-association participant SD
PROFILE_FOLD_HEMISPHERES = True
PROFILE_TARGET_HEMISPHERE = 'left'
PROFILE_DISTANCE_BINS_MM = list(range(0, 101, 10))
PROFILE_PERMUTATIONS = 199
PROFILE_BOOTSTRAPS = 500
PROFILE_MIN_NORM_QUANTILE = .1

config['analyses']['pairing_profiles'] = dict(n_components=PROFILE_N_COMPONENTS,
    spatial_scope='within_participant', fold_hemispheres=PROFILE_FOLD_HEMISPHERES,
    target_hemisphere=PROFILE_TARGET_HEMISPHERE, bin_edges=PROFILE_DISTANCE_BINS_MM,
    n_permutations=PROFILE_PERMUTATIONS, n_bootstrap=PROFILE_BOOTSTRAPS,
    min_norm_quantile=PROFILE_MIN_NORM_QUANTILE)
profile_spatial = prepare_analysis(config, 'pairing_profiles',
    OUTPUT_DIR / 'pairing_profiles_config.json', RECOMPUTE)


In [ ]:
FIGSIZE = (19, 5)
if profile_spatial is not None:
    fig, axes = plt.subplots(1, 3, figsize=FIGSIZE, layout='constrained')
    for modality, color in MODALITY_COLORS.items():
        across = profile_spatial['summary'].query('modality == @modality')
        selected = PROFILE_ASSOCIATION - 1
        within = profile_spatial['iteration_curves'].query('modality == @modality and iteration == @selected')
        for ax, table, metric, error, null in [
            (axes[0], across, 'similarity_mean', 'similarity_association_sd', 'null_mean_mean'),
            (axes[1], within, 'similarity', 'similarity_subject_sd', 'null_mean'),
            (axes[2], across, 'excess_similarity_mean', 'excess_similarity_association_sd', None)]:
            ax.plot(table.distance_mm, table[metric], 'o-', color=color, label=modality)
            ax.fill_between(table.distance_mm, table[metric]-table[error], table[metric]+table[error], color=color, alpha=.18)
            if null is not None:
                ax.plot(table.distance_mm, table[null], '--', color=color)
    for ax, title, label in zip(axes,
        ['Across associations: mean ± SD', f'Association {PROFILE_ASSOCIATION}: participant mean ± SD', 'Excess similarity: mean ± association SD'],
        ['Absolute-profile cosine similarity', 'Absolute-profile cosine similarity', 'Cosine similarity minus spatial-shuffle mean']):
        ax.set(title=title, xlabel='Distance (mm; matched bins)', ylabel=label)
        ax.grid(alpha=.2)
        ax.legend()
    axes[2].axhline(0, color='gray', linestyle='--')
    plt.show()


In [ ]:
FIGSIZE = (13, 5)
if profile_spatial is not None:
    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, layout='constrained')
    for modality, color in MODALITY_COLORS.items():
        across = profile_spatial['summary'].query('modality == @modality')
        selected = PROFILE_ASSOCIATION - 1
        within = profile_spatial['iteration_curves'].query('modality == @modality and iteration == @selected')
        for ax, table, metric, error, null in [
            (axes[0], across, 'magnitude_semivariance_mean', 'magnitude_semivariance_association_sd', 'magnitude_null_mean'),
            (axes[1], within, 'magnitude_semivariance', 'magnitude_semivariance_subject_sd', 'magnitude_null')]:
            ax.plot(table.distance_mm, table[metric], 'o-', color=color, label=modality)
            ax.fill_between(table.distance_mm, table[metric]-table[error], table[metric]+table[error], color=color, alpha=.18)
            ax.plot(table.distance_mm, table[null], '--', color=color)
    for ax, title in zip(axes, ['Across associations: mean ± SD', f'Association {PROFILE_ASSOCIATION}: participant mean ± SD']):
        ax.set(title=title, xlabel='Distance (mm; matched bins)', ylabel='Normalized magnitude semivariance')
        ax.grid(alpha=.2)
        ax.legend()
    plt.show()


## Gradients
Amplitude is the unthresholded norm of the first `GRADIENT_N_COMPONENTS` fitted weights:
$A_i=\sqrt{\sum_{c=1}^{k}w_{ic}^2}$. These are model-weight magnitudes, not measured neural activity.

`GRADIENT_SCOPE='pooled'` combines all coordinates; `'within_subject'` uses each participant's features from the same group fit. Coordinates are MNI millimeters. `GRADIENT_FOLD_HEMISPHERES=True` reflects all coordinates onto `GRADIENT_TARGET_HEMISPHERE` (left or right) before peak selection, distance computation, and fitting; `False` uses original coordinates. Folding retains all features and weights, including coincident reflected locations, and stores their original coordinates in `original_x/y/z`. Folded distances describe hemispheric alignment, not physical interhemispheric separation. Zero-distance observations remain visible but are excluded from fitting. Distances are Euclidean, not cortical geodesic distances. Pooling weights each feature equally and therefore reflects unequal sampling coverage.

A numeric `GRADIENT_RADIUS_MM` identifies maxima with no larger norm in that neighborhood; equal values favor the earlier feature row. Isolated features without another neighbor are excluded. Set the radius to `None` to select the top `GRADIENT_TOP_PERCENT` percent instead (count rounded upward, ties resolved by row, zero norms excluded). In this mode, selected electrodes need not be spatial local maxima. `GRADIENT_MAX_PLOTS=None` plots all selected maxima; an integer limits plotted/fitted maxima, while the regional count plot retains all detected maxima.

Select `GRADIENT_FIT_MODEL` at the top of the configuration cell: `linear`, `quadratic`, `gaussian`, or `exponential`. The models are $b+md$, $b-cd^2$ ($c\geq0$), $b+a\exp[-d^2/(2\sigma^2)]$, and $b+a\exp(-d/\tau)$, respectively. Gaussian and exponential fits constrain baseline and amplitude to be nonnegative; the linear fit is unconstrained. The quadratic vertex is fixed at distance zero, with fitted height $b$; the observed peak remains excluded from fitting. Flat boundary solutions are flagged as `no_decay_boundary` and excluded from valid-fit summaries. Rerun analysis after changing this selection. Only points within `GRADIENT_FIT_RADIUS_MM` contribute to fitting; zero-distance points, including the selected maximum, are excluded. Scatter plots show all features in the scope. Each modality has one magnitude brain map using Reds and one radial plot containing all selected peaks.

R² and RMSE describe in-sample fit quality, not a formal test of Gaussian spatial organization. Peak selection, spatial dependence, uneven coverage, and overlapping peaks limit interpretation. The fit is exploratory; inspect the scatter and fit status as well as R².

All subjects and peaks are overlaid in one radial plot per modality. Points and individual fitted curves use the shared modality colors. In within-subject mode, the black mean first averages valid peak curves within each subject and then averages subjects equally. Its band is ±1 sample SD across subject mean curves. In pooled mode, the mean and SD weight valid peak curves equally. Predictions for the summary use the intersection of all valid observed fitting-distance ranges, without extrapolation. Failed or boundary fits retain their points but do not contribute curves; fewer than two contributing units yields no SD band. These are descriptive summaries, not confidence intervals; selected peaks can share observations. The average fitted curve need not itself belong to the selected model family. Pooled data without native subject identity are labeled Unassigned or retain their aggregate label, rather than inventing subject identities.


In [ ]:
# must have teh variability of the rdn pairing
INPUT_DIR = result_directory(config, 'weights')
OUTPUT_DIR = Path(config['output_dir'])
RECOMPUTE = False
GRADIENT_FIT_MODEL = 'quadratic'
GRADIENT_SCOPE = 'within_subject'
GRADIENT_FOLD_HEMISPHERES = True
GRADIENT_TARGET_HEMISPHERE = 'left'
GRADIENT_N_COMPONENTS = 2
GRADIENT_RADIUS_MM = None
GRADIENT_TOP_PERCENT = 1
GRADIENT_FIT_RADIUS_MM = 30
GRADIENT_MAX_PLOTS = None
GRADIENT_MODALITIES = ('iEEG', 'MEG')

config['analyses']['gradients'] = dict(n_components=GRADIENT_N_COMPONENTS, spatial_scope=GRADIENT_SCOPE, neighborhood_mm=GRADIENT_RADIUS_MM, top_percent=GRADIENT_TOP_PERCENT, max_peaks=GRADIENT_MAX_PLOTS, fit_radius_mm=GRADIENT_FIT_RADIUS_MM, fold_hemispheres=GRADIENT_FOLD_HEMISPHERES, target_hemisphere=GRADIENT_TARGET_HEMISPHERE, fit_model=GRADIENT_FIT_MODEL)
gradient_results = prepare_analysis(config, 'gradients', OUTPUT_DIR / 'gradients_config.json', RECOMPUTE)


In [ ]:
# keep the color like before + start for maximum in the brain plot only don't show the other

FIGSIZE = (13, 5)
from vs3_1_gradients import predict_gradient_fit
if gradient_results is not None:
    for modality, scopes in gradient_results.items():
        fig, axes = plt.subplots(1, 2, figsize=FIGSIZE, layout='constrained')
        features = pd.concat([r['features'] for r in scopes.values()], ignore_index=True)
        plotting.plot_markers(features.weight_norm.to_numpy(), features[['x', 'y', 'z']].to_numpy(),
            axes=axes[0], figure=fig, node_cmap='Reds', node_vmin=0, node_size=3,
            title=f'{modality}: absolute-weight norm', display_mode='ortho')
        entries = []
        for scope, result in scopes.items():
            for peak_id, profile in result['profiles'].items():
                axes[1].scatter(profile.distance_mm, profile.weight_norm, s=3, alpha=.1,
                                color=MODALITY_COLORS[modality], rasterized=True)
                fit = result['fits'].set_index('peak_id').loc[peak_id]
                distances = profile.loc[profile.in_fit_radius, 'distance_mm']
                if fit.fit_status == 'ok' and len(distances) and distances.min() < distances.max():
                    d = np.linspace(distances.min(), distances.max(), 200)
                    axes[1].plot(d, predict_gradient_fit(d, fit, result['fit_model']),
                                 color=MODALITY_COLORS[modality], alpha=.3)
                    entries.append((scope, fit, distances.min(), distances.max()))
        if entries and max(e[2] for e in entries) < min(e[3] for e in entries):
            d = np.linspace(max(e[2] for e in entries), min(e[3] for e in entries), 200)
            curves = np.array([predict_gradient_fit(d, e[1], GRADIENT_FIT_MODEL) for e in entries])
            if GRADIENT_SCOPE == 'within_subject':
                owners = np.array([e[0] for e in entries])
                curves = np.array([curves[owners == owner].mean(axis=0) for owner in dict.fromkeys(owners)])
            mean = curves.mean(axis=0)
            axes[1].plot(d, mean, color='black', lw=2, label='Mean fitted profile')
            if len(curves) > 1:
                sd = curves.std(axis=0, ddof=1)
                axes[1].fill_between(d, mean-sd, mean+sd, color='gray', alpha=.2, label='±1 SD')
            axes[1].legend()
        axes[1].set(xlabel='Distance from peak (mm)', ylabel='Component weight norm', title=f'{modality} · {GRADIENT_FIT_MODEL}')
        axes[1].grid(alpha=.2)
        plt.show()


In [ ]:
FIGSIZE = (14, 5)
if gradient_results is not None:
    peak_frames = []
    for modality, scopes in gradient_results.items():
        for scope, result in scopes.items():
            peak_frames.append(result['maxima'].assign(modality=modality))
    peak_table = pd.concat(peak_frames, ignore_index=True)
    counts = peak_table.groupby(['region_shared', 'modality']).size().unstack(fill_value=0)
    fig, ax = plt.subplots(figsize=FIGSIZE, layout='constrained')
    positions = np.arange(len(counts))
    for offset, modality in enumerate(['iEEG', 'MEG']):
        values = counts.get(modality, pd.Series(0, index=counts.index))
        ax.bar(positions + (offset-.5)*.4, values, width=.4,
               color=MODALITY_COLORS[modality], label=modality)
    ax.set(xticks=positions, xticklabels=counts.index, ylabel='Detected weight-norm maxima')
    ax.tick_params(axis='x', rotation=90)
    ax.grid(axis='y', alpha=.2)
    ax.legend()
    plt.show()


In [ ]:
FIGSIZE = (15, 5)

selected_gradient_metrics = None
if gradient_results is not None:
    frames = []
    for modality, scopes in gradient_results.items():
        for scope, result in scopes.items():
            if result['fits'].empty:
                continue
            frame = result['fits'].copy()
            frame['model'] = result['fit_model']
            frame['modality'], frame['scope'] = modality, scope
            frame['within_subject'] = not result['features'].scope.eq('pooled').all()
            frame['matched_peak'] = frame.fit_status.eq('ok')
            frames.append(frame)
    if frames:
        selected_gradient_metrics = pd.concat(frames, ignore_index=True)
        display(selected_gradient_metrics.groupby(['modality', 'fit_status']).size())
from vs3_1_gradients import GRADIENT_PARAMETERS
if selected_gradient_metrics is not None:
    table = selected_gradient_metrics
    models = (GRADIENT_FIT_MODEL,)
    title = f'{MODEL} · radial model comparison'
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    models = tuple(models)
    modalities = list(table.modality.unique()) if not table.empty else ['iEEG', 'MEG']
    colors = {m: {'iEEG': 'navy', 'MEG': 'orange'}.get(m, f'C{i}') for i, m in enumerate(modalities)}
    metrics = ['adjusted_r_squared'] + [GRADIENT_PARAMETERS[m][0] for m in models]
    units = []
    if not table.empty:
        for (modality, within), frame in table[table.matched_peak].groupby(['modality', 'within_subject']):
            if within:
                frame = frame.groupby(['scope', 'model'])[metrics].median().reset_index()
            else:
                frame = frame.copy()
            frame['modality'] = modality
            units.append(frame)
    data = pd.concat(units, ignore_index=True) if units else pd.DataFrame()
    rng = np.random.default_rng(2026)

    def draw(ax, values, position, color):
        values = np.asarray(values, float)
        values = values[np.isfinite(values)]
        if len(values):
            ax.boxplot([values], positions=[position], widths=0.22, showfliers=False, manage_ticks=False, patch_artist=True, boxprops=dict(facecolor=color, alpha=0.18), medianprops=dict(color=color, linewidth=2))
            ax.scatter(position + rng.uniform(-0.07, 0.07, len(values)), values, s=8, alpha=0.5, color=color, rasterized=True)
        ax.text(position, 0.02, f'n={len(values)}', transform=ax.get_xaxis_transform(), ha='center', fontsize=8, color=color)
    fig, combined_axes = plt.subplots(1, 2, figsize=FIGSIZE, layout='constrained')
    ax = combined_axes[0]
    for i, model in enumerate(models):
        for j, modality in enumerate(modalities):
            position = i + (j - (len(modalities) - 1) / 2) * 0.3
            values = data.loc[(data.model == model) & (data.modality == modality), 'adjusted_r_squared'] if not data.empty else []
            draw(ax, values, position, colors[modality])
    ax.set(xticks=range(len(models)), xticklabels=[m.capitalize() for m in models], ylabel='Adjusted R² (in-sample)', title=f'{title} · goodness of fit', xlim=(-0.6, len(models) - 0.4))
    ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')
    ax.legend(handles=[Line2D([], [], marker='o', linestyle='', color=colors[m], label=m) for m in modalities])
    ax.grid(axis='y', alpha=0.15)

    figures = {'goodness_of_fit': fig}
    for model in models:
        metric, label = GRADIENT_PARAMETERS[model]
        ax = combined_axes[1]
        for j, modality in enumerate(modalities):
            values = data.loc[(data.model == model) & (data.modality == modality), metric] if not data.empty else []
            draw(ax, values, j, colors[modality])
        ax.set(xticks=range(len(modalities)), xticklabels=modalities, ylabel=label, title=f'{title} · {model.capitalize()}', xlim=(-0.6, len(modalities) - 0.4))
        ax.grid(axis='y', alpha=0.15)
        figures[f'{model}_parameter'] = fig
    plt.show()



### Exploratory selection of gradient radii
The printed cluster command searches `GRADIENT_RADIUS_MM` (peak-selection neighborhood) and `GRADIENT_FIT_RADIUS_MM` (point-inclusion radius) on editable 10–100 mm grids. The figure has one panel per modality: **x = inclusion radius, y = peak-selection radius, color = mean adjusted R² for the selected model**. Color limits are shared across modalities; all finite estimates are colored; × marks limited support, and gray means the metric is undefined. White stars identify suggested pairs without changing the main Gradient settings. The cell has independent scope, component, hemisphere, and support settings. Numeric neighborhood radii select local maxima rather than the top-percentage mode.

**Selected-model fit quality.** Choose `RADIUS_FIT_MODEL` at the top of the cell: `linear` ($b+md$), `quadratic` ($b-cd^2$, $c\geq0$, second order with its maximum at the peak), `gaussian` ($b+a\exp[-d^2/(2\sigma^2)]$), or `exponential` ($b+a\exp[-d/\tau]$). Maximize mean $R^2_{adj}=1-(1-R^2)(n-1)/(n-p)$, with $p=2$ for linear/quadratic and $p=3$ for Gaussian/exponential. The linear slope is unconstrained; Gaussian/exponential baseline and amplitude are nonnegative. Negative scores are retained. Exact ties favor smaller neighborhood and inclusion radii. This is in-sample optimization across different domains and peak sets, not a validated biological optimum; small domains can be easier to fit.

The metric averages valid peaks equally in pooled analysis. Within-subject analysis first averages peaks within each subject, then averages subjects equally. For automatic recommendations, defaults require at least three peaks with six or more observations and 80% coverage by those supported peaks. Exploratory evaluations use all computable peaks: at least three observations at two or more distinct distances for linear/quadratic; Gaussian/exponential retain six observations at four or more distinct distances for nonlinear-fit stability. Limited-support estimates are displayed with × but are excluded from automatic recommendations. Constant magnitudes, absence of selected peaks, or insufficient observations can still leave values undefined; flat no-decay quadratic solutions remain excluded. Support tables include peak counts, subject/peak analysis-unit counts, and point counts. Peak identities and support change across radius pairs, and neighborhoods overlap. Inspect counts and boundary selections before copying suggested values into the main Gradient configuration. Modalities can suggest different pairs.


In [ ]:
INPUT_DIR = result_directory(config, 'weights')
OUTPUT_DIR = Path(config['output_dir'])
RECOMPUTE = False
RADIUS_FIT_MODEL = 'quadratic'
RADIUS_NEIGHBORHOOD_RADII_MM = np.arange(10.0, 101.0, 10.0)
RADIUS_FIT_RADII_MM = np.arange(10.0, 101.0, 10.0)
RADIUS_SCOPE = 'within_subject'
RADIUS_N_COMPONENTS = 2
RADIUS_FOLD_HEMISPHERES = True
RADIUS_TARGET_HEMISPHERE = 'left'
RADIUS_MODALITIES = ('iEEG', 'MEG')
RADIUS_MAX_PEAKS = None
RADIUS_MIN_POINTS = 6
RADIUS_MIN_PEAKS = 3
RADIUS_MIN_VALID_FRACTION = 0.8

config['analyses']['radius_search'] = dict(neighborhood_radii=RADIUS_NEIGHBORHOOD_RADII_MM.tolist(), fit_radii=RADIUS_FIT_RADII_MM.tolist(), metric=RADIUS_FIT_MODEL, n_components=RADIUS_N_COMPONENTS, spatial_scope=RADIUS_SCOPE, fold_hemispheres=RADIUS_FOLD_HEMISPHERES, target_hemisphere=RADIUS_TARGET_HEMISPHERE, max_peaks=RADIUS_MAX_PEAKS, min_points=RADIUS_MIN_POINTS, min_peaks=RADIUS_MIN_PEAKS, min_valid_fraction=RADIUS_MIN_VALID_FRACTION)
radius_search = prepare_analysis(config, 'radius_search', OUTPUT_DIR / 'radius_search_config.json', RECOMPUTE)


In [ ]:
if radius_search is not None:
    results, candidates = radius_search['results'], radius_search['candidates']
    title = 'Radius search'
    import matplotlib.pyplot as plt
    from matplotlib.colors import Normalize

    def edges(x):
        x = np.asarray(x, float)
        if len(x) == 1:
            return np.array([x[0] - 0.5, x[0] + 0.5])
        mid = (x[:-1] + x[1:]) / 2
        return np.r_[x[0] - (x[1] - x[0]) / 2, mid, x[-1] + (x[-1] - x[-2]) / 2]
    fig, axes = plt.subplots(1, len(results), figsize=(7 * len(results), 6), squeeze=False, layout='constrained', sharex=True, sharey=True)
    metrics = {r['grid'].metric.iloc[0] for r in results.values()}
    if len(metrics) != 1:
        raise ValueError('Use one metric for all panels.')
    metric = next(iter(metrics))
    selection_column = 'top_percent' if 'top_percent' in next(iter(results.values()))['grid'].columns else 'neighborhood_mm'
    selection_label = 'Top-magnitude features selected (%)' if selection_column == 'top_percent' else 'Peak-selection neighborhood radius (mm)'
    values = np.concatenate([r['grid']['value'].to_numpy() for r in results.values()])
    values = values[np.isfinite(values)]
    lo, hi = (float(values.min()), float(values.max())) if len(values) else (0.0, 1.0)
    if lo == hi:
        hi = lo + max(1e-06, abs(lo) * 0.01)
    norm = Normalize(lo, hi)
    for ax, (modality, result) in zip(axes.flat, results.items()):
        grid = result['grid'].copy()
        matrix = grid.pivot(index=selection_column, columns='fit_radius_mm', values='value').sort_index().sort_index(axis=1)
        ax.set_facecolor('#eeeeee')
        mesh = ax.pcolormesh(edges(matrix.columns), edges(matrix.index), np.ma.masked_invalid(matrix.to_numpy()), cmap='viridis', norm=norm)
        low_support = grid.loc[~grid.eligible & np.isfinite(grid.value)]
        if len(low_support):
            ax.scatter(low_support.fit_radius_mm, low_support[selection_column], marker='x', s=30, color='black', linewidths=0.8, label='Limited support')
        chosen = candidates.get(modality) if candidates else None
        if chosen is not None and (not chosen.empty):
            if metric == 'semivariance':
                ax.plot(chosen.fit_radius_mm, chosen.neighborhood_mm, 'wo', markersize=4, label='Plateau candidates')
            best = chosen.iloc[0]
            ax.scatter(best.fit_radius_mm, best[selection_column], marker='*', s=200, facecolor='white', edgecolor='black', label='Suggested pair')
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=8, loc='upper right')
        ax.set(title=modality, xlabel='Fitting / inclusion radius (mm)', ylabel=selection_label, xticks=matrix.columns, yticks=matrix.index)
        if not np.isfinite(matrix.to_numpy()).any():
            ax.text(0.5, 0.5, 'No computable estimates', transform=ax.transAxes, ha='center')
    label = 'Normalized local magnitude semivariance' if metric == 'semivariance' else f'Mean adjusted R² of {metric} fits'
    fig.colorbar(mesh, ax=list(axes.flat), label=label, shrink=0.85)
    fig.suptitle(title or f'{label} · × = limited support; gray = undefined')
    plt.show()


### Exploratory selection of top percentage and fitting radius
This independent analysis uses **`GRADIENT_RADIUS_MM=None`** and searches the top percentage of weight magnitudes together with the fitting radius. The default percentage grid is 0.5, 1, 2, 3, 5, 7.5, and 10%; the fitting-radius grid is 10–100 mm. Edit both grids below. Each figure has one panel per modality: **x = fitting radius, y = top percentage, color = mean adjusted R²**. The existing 0–1 color scale is retained; negative scores remain in the tables and use the lowest color.

Top percentages are applied within each analysis scope, with the selected count rounded upward and zero magnitudes excluded. These selected features are not necessarily spatial local maxima. The fit model, hemisphere settings, scope, and support rules are independent of the preceding cell. All finite estimates are displayed; × marks limited support and gray marks undefined estimates. Recommendations maximize supported mean adjusted R², with smaller percentages and radii breaking exact ties. Within-subject results average peaks within subjects and then average subjects equally.

Suggested settings are not automatically applied to the main Gradient configuration. To use them, set `GRADIENT_RADIUS_MM=None`, then copy the recommended `top_percent` and `fit_radius_mm` values. Adjacent percentage settings may select identical features in small datasets. Keep `PERCENT_SEARCH_MAX_PEAKS=None` for the complete percentage search; a cap can make distinct percentages select the same features. This is exploratory in-sample selection over changing feature subsets, not independent validation.


In [ ]:
INPUT_DIR = result_directory(config, 'weights')
OUTPUT_DIR = Path(config['output_dir'])
RECOMPUTE = False
PERCENT_SEARCH_FIT_MODEL = 'exponential'
PERCENT_SEARCH_TOP_PERCENTS = np.array([0.5, 1.0, 2.0, 3.0, 5.0, 7.5, 10.0])
PERCENT_SEARCH_FIT_RADII_MM = np.arange(10.0, 101.0, 10.0)
PERCENT_SEARCH_SCOPE = 'within_subject'
PERCENT_SEARCH_N_COMPONENTS = 2
PERCENT_SEARCH_FOLD_HEMISPHERES = True
PERCENT_SEARCH_TARGET_HEMISPHERE = 'left'
PERCENT_SEARCH_MODALITIES = ('iEEG', 'MEG')
PERCENT_SEARCH_MAX_PEAKS = None
PERCENT_SEARCH_MIN_POINTS = 6
PERCENT_SEARCH_MIN_PEAKS = 3
PERCENT_SEARCH_MIN_VALID_FRACTION = 0.8

config['analyses']['percent_search'] = dict(top_percents=PERCENT_SEARCH_TOP_PERCENTS.tolist(), fit_radii=PERCENT_SEARCH_FIT_RADII_MM.tolist(), metric=PERCENT_SEARCH_FIT_MODEL, n_components=PERCENT_SEARCH_N_COMPONENTS, spatial_scope=PERCENT_SEARCH_SCOPE, fold_hemispheres=PERCENT_SEARCH_FOLD_HEMISPHERES, target_hemisphere=PERCENT_SEARCH_TARGET_HEMISPHERE, max_peaks=PERCENT_SEARCH_MAX_PEAKS, min_points=PERCENT_SEARCH_MIN_POINTS, min_peaks=PERCENT_SEARCH_MIN_PEAKS, min_valid_fraction=PERCENT_SEARCH_MIN_VALID_FRACTION)
percent_search = prepare_analysis(config, 'percent_search', OUTPUT_DIR / 'percent_search_config.json', RECOMPUTE)


In [ ]:
if percent_search is not None:
    results, candidates = percent_search['results'], percent_search['candidates']
    title = 'Percent search'
    import matplotlib.pyplot as plt
    from matplotlib.colors import Normalize

    def edges(x):
        x = np.asarray(x, float)
        if len(x) == 1:
            return np.array([x[0] - 0.5, x[0] + 0.5])
        mid = (x[:-1] + x[1:]) / 2
        return np.r_[x[0] - (x[1] - x[0]) / 2, mid, x[-1] + (x[-1] - x[-2]) / 2]
    fig, axes = plt.subplots(1, len(results), figsize=(7 * len(results), 6), squeeze=False, layout='constrained', sharex=True, sharey=True)
    metrics = {r['grid'].metric.iloc[0] for r in results.values()}
    if len(metrics) != 1:
        raise ValueError('Use one metric for all panels.')
    metric = next(iter(metrics))
    selection_column = 'top_percent' if 'top_percent' in next(iter(results.values()))['grid'].columns else 'neighborhood_mm'
    selection_label = 'Top-magnitude features selected (%)' if selection_column == 'top_percent' else 'Peak-selection neighborhood radius (mm)'
    values = np.concatenate([r['grid']['value'].to_numpy() for r in results.values()])
    values = values[np.isfinite(values)]
    lo, hi = (float(values.min()), float(values.max())) if len(values) else (0.0, 1.0)
    if lo == hi:
        hi = lo + max(1e-06, abs(lo) * 0.01)
    norm = Normalize(lo, hi)
    for ax, (modality, result) in zip(axes.flat, results.items()):
        grid = result['grid'].copy()
        matrix = grid.pivot(index=selection_column, columns='fit_radius_mm', values='value').sort_index().sort_index(axis=1)
        ax.set_facecolor('#eeeeee')
        mesh = ax.pcolormesh(edges(matrix.columns), edges(matrix.index), np.ma.masked_invalid(matrix.to_numpy()), cmap='viridis', norm=norm)
        low_support = grid.loc[~grid.eligible & np.isfinite(grid.value)]
        if len(low_support):
            ax.scatter(low_support.fit_radius_mm, low_support[selection_column], marker='x', s=30, color='black', linewidths=0.8, label='Limited support')
        chosen = candidates.get(modality) if candidates else None
        if chosen is not None and (not chosen.empty):
            if metric == 'semivariance':
                ax.plot(chosen.fit_radius_mm, chosen.neighborhood_mm, 'wo', markersize=4, label='Plateau candidates')
            best = chosen.iloc[0]
            ax.scatter(best.fit_radius_mm, best[selection_column], marker='*', s=200, facecolor='white', edgecolor='black', label='Suggested pair')
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=8, loc='upper right')
        ax.set(title=modality, xlabel='Fitting / inclusion radius (mm)', ylabel=selection_label, xticks=matrix.columns, yticks=matrix.index)
        if not np.isfinite(matrix.to_numpy()).any():
            ax.text(0.5, 0.5, 'No computable estimates', transform=ax.transAxes, ha='center')
    label = 'Normalized local magnitude semivariance' if metric == 'semivariance' else f'Mean adjusted R² of {metric} fits'
    fig.colorbar(mesh, ax=list(axes.flat), label=label, shrink=0.85)
    fig.suptitle(title or f'{label} · × = limited support; gray = undefined')
    plt.show()


### Compare radial models: linear, quadratic, Gaussian, and exponential
This cell has its own configuration below, independent of the earlier Gradient analysis. It recomputes peak selection and radial profiles from the existing fitted `weights` and `datasets`, using its own modalities, component count, scope, hemisphere folding/target, and fitting radius. Change these settings and rerun this cell alone. Zero-distance observations remain excluded. No group model is refitted. Hemisphere folding reflects both hemispheres onto the selected side; it does not restrict inclusion to that hemisphere.

For weight norm $A$ and distance $d$ (mm), fit:

| Model | Function | Separate parameter plot |
|---|---|---|
| Linear | $A(d)=b+md$ | Slope $m$ (weight norm/mm) |
| Quadratic | $A(d)=b-cd^2$, $c\geq0$ | Curvature $-2c$ (weight norm/mm²) |
| Gaussian | $A(d)=b+a\exp[-d^2/(2\sigma^2)]$ | Width $\sigma$ (mm) |
| Exponential | $A(d)=b+a\exp(-d/\tau)$ | Decay length $\tau$ (mm) |

The exponential allows an immediate decrease at the origin, whereas the Gaussian has zero slope there. The quadratic is constrained to have its maximum at distance zero and decrease for positive distances when $c>0$. Its height $b$ is fitted rather than fixed to the excluded observed peak. The $c=0$ boundary is flagged as `no_decay_boundary` and excluded from valid-fit summaries. Do not extrapolate beyond the fitted radius. Gaussian/exponential baseline and amplitude are nonnegative; the linear coefficients are unconstrained, and quadratic curvature is nonpositive.

The combined goodness-of-fit plot uses $R^2_{\mathrm{adj}}=1-(1-R^2)(n-1)/(n-p)$, where $R^2=1-\mathrm{SSE}/\mathrm{SST}$ and $p$ counts fitted coefficients including the intercept (2 for linear and the constrained quadratic, 3 for Gaussian and exponential). Higher values indicate better descriptive fit after a parameter-count adjustment; negative values are retained. This remains an in-sample comparison, not spatial cross-validation or a significance test.

All five figures use **matched peaks with valid fits for every requested model**; the status table reports exclusions. Dots represent individual peaks for pooled results and one median per subject for within-subject results. Boxes show medians and quartiles, with 1.5-IQR whiskers. Parameter plots characterize shape/scale, not relationship strength: larger σ or τ means slower decay; negative linear slope means decreasing amplitude; more negative quadratic curvature means faster decline away from the fixed vertex. Raw slopes/curvatures depend on weight scaling, so modality differences require caution. Peaks share observations and are not independent.

Fitting uses [SciPy nonlinear least squares](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html) for decay models and linear least squares for polynomial models. See also the [definition of R²](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html).


In [ ]:
INPUT_DIR = result_directory(config, 'weights')
OUTPUT_DIR = Path(config['output_dir'])
RECOMPUTE = False
GRADIENT_COMPARISON_SCOPE = 'within_subject'
GRADIENT_COMPARISON_FOLD_HEMISPHERES = True
GRADIENT_COMPARISON_TARGET_HEMISPHERE = 'left'
GRADIENT_COMPARISON_N_COMPONENTS = 2
GRADIENT_COMPARISON_RADIUS_MM = None #90
GRADIENT_COMPARISON_TOP_PERCENT = 1
GRADIENT_COMPARISON_FIT_RADIUS_MM = 30.0
GRADIENT_COMPARISON_MAX_PEAKS = None
GRADIENT_COMPARISON_MODALITIES = ('iEEG', 'MEG')
GRADIENT_COMPARISON_MODELS = ('linear', 'quadratic', 'gaussian', 'exponential')

config['analyses']['gradient_comparison'] = dict(n_components=GRADIENT_COMPARISON_N_COMPONENTS, spatial_scope=GRADIENT_COMPARISON_SCOPE, neighborhood_mm=GRADIENT_COMPARISON_RADIUS_MM, top_percent=GRADIENT_COMPARISON_TOP_PERCENT, max_peaks=GRADIENT_COMPARISON_MAX_PEAKS, fit_radius_mm=GRADIENT_COMPARISON_FIT_RADIUS_MM, fold_hemispheres=GRADIENT_COMPARISON_FOLD_HEMISPHERES, target_hemisphere=GRADIENT_COMPARISON_TARGET_HEMISPHERE, models=list(GRADIENT_COMPARISON_MODELS))
gradient_model_comparison = prepare_analysis(config, 'gradient_comparison', OUTPUT_DIR / 'gradient_comparison_config.json', RECOMPUTE)


In [ ]:
PARAMETER_FIGSIZE = (14, 10)

from vs3_1_gradients import GRADIENT_PARAMETERS
if gradient_model_comparison is not None:
    table = gradient_model_comparison
    models = GRADIENT_COMPARISON_MODELS
    title = f'{MODEL} · radial model comparison'
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    models = tuple(models)
    modalities = list(table.modality.unique()) if not table.empty else ['iEEG', 'MEG']
    colors = {m: {'iEEG': 'navy', 'MEG': 'orange'}.get(m, f'C{i}') for i, m in enumerate(modalities)}
    metrics = ['adjusted_r_squared'] + [GRADIENT_PARAMETERS[m][0] for m in models]
    units = []
    if not table.empty:
        for (modality, within), frame in table[table.matched_peak].groupby(['modality', 'within_subject']):
            if within:
                frame = frame.groupby(['scope', 'model'])[metrics].median().reset_index()
            else:
                frame = frame.copy()
            frame['modality'] = modality
            units.append(frame)
    data = pd.concat(units, ignore_index=True) if units else pd.DataFrame()
    rng = np.random.default_rng(2026)

    def draw(ax, values, position, color):
        values = np.asarray(values, float)
        values = values[np.isfinite(values)]
        if len(values):
            ax.boxplot([values], positions=[position], widths=0.22, showfliers=False, manage_ticks=False, patch_artist=True, boxprops=dict(facecolor=color, alpha=0.18), medianprops=dict(color=color, linewidth=2))
            ax.scatter(position + rng.uniform(-0.07, 0.07, len(values)), values, s=8, alpha=0.5, color=color, rasterized=True)
        ax.text(position, 0.02, f'n={len(values)}', transform=ax.get_xaxis_transform(), ha='center', fontsize=8, color=color)
    fig, ax = plt.subplots(figsize=(11, 5), layout='constrained')
    for i, model in enumerate(models):
        for j, modality in enumerate(modalities):
            position = i + (j - (len(modalities) - 1) / 2) * 0.3
            values = data.loc[(data.model == model) & (data.modality == modality), 'adjusted_r_squared'] if not data.empty else []
            draw(ax, values, position, colors[modality])
    ax.set(xticks=range(len(models)), xticklabels=[m.capitalize() for m in models], ylabel='Adjusted R² (in-sample)', title=f'{title} · goodness of fit', xlim=(-0.6, len(models) - 0.4))
    ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')
    ax.legend(handles=[Line2D([], [], marker='o', linestyle='', color=colors[m], label=m) for m in modalities])
    ax.grid(axis='y', alpha=0.15)

    figures = {'goodness_of_fit': fig}
    parameter_fig, parameter_axes = plt.subplots(2, 2, figsize=PARAMETER_FIGSIZE, layout='constrained')
    for ax, model in zip(parameter_axes.flat, models):
        metric, label = GRADIENT_PARAMETERS[model]
        for j, modality in enumerate(modalities):
            values = data.loc[(data.model == model) & (data.modality == modality), metric] if not data.empty else []
            draw(ax, values, j, colors[modality])
        ax.set(xticks=range(len(modalities)), xticklabels=modalities, ylabel=label, title=f'{title} · {model.capitalize()}', xlim=(-0.6, len(modalities) - 0.4))
        ax.grid(axis='y', alpha=0.15)
        figures[f'{model}_parameter'] = parameter_fig
    plt.show()


## Clustering absolute model weights
Clustering uses **absolute weights of the first `WEIGHT_N_COMPONENTS` components**, separately for each modality, without spatial coordinates or axis standardization. Thresholding is **off by default**. When enabled, magnitudes below the component-specific mean + population SD are set to zero. All-zero profiles are excluded (label −1) and shown gray on the brain. Component signs are discarded; cluster labels are not matched across methods or modalities.

Export the existing fitted weights once below. Run the printed command in a terminal or cluster job: all cluster fitting and summary computation occur in `vs3_1_weight_clustering.py`. No group-model refit is needed. Subsequent plotting only loads saved results. Re-export and recompute if the model or its input data change.

Available methods: **kmeans** (Euclidean partitions), **minibatch** (scalable approximate K-means), **gmm** (full-covariance Gaussian mixture, hard maximum-probability assignments), and **birch** (hierarchical compression followed by clustering). BIRCH may return fewer clusters than requested. Its threshold is a fraction of the RMS input-vector norm (default 0.1). All methods use the same magnitude geometry; cluster count is exploratory.

Automatic selection (`WEIGHT_N_CLUSTERS = "auto"`) fits candidate counts 2–10 separately for each modality and method, choosing the highest Euclidean silhouette score. Fits use all nonzero profiles; scores use a fixed seeded sample of at most 2,000 features to limit memory. Exact ties favor smaller counts. Inspect the saved score curve, particularly boundary selections. This criterion favors compact, separated groups and does not test whether clustering exists; it can also be less appropriate for elongated Gaussian-mixture clusters. Use an integer to retain manual selection.

`WEIGHT_N_COMPONENTS` controls both fitting and silhouette scoring. Export includes all fitted components, so changing this setting requires only rerunning clustering, not the group model (provided enough components were fitted). The weight-space figure is 2D for two components and 3D for three or more components (displaying only the first three axes). For one component, the second display axis is zero. Results are stored separately by component count. Re-export once if using older three-component exports.


In [ ]:
INPUT_DIR = result_directory(config, 'weights')
OUTPUT_DIR = INPUT_DIR / 'absolute_weight_clustering'
from vs3_1_weight_clustering import export_inputs, load_results
import shlex
WEIGHT_COMPONENT_COUNTS = list(range(2, N_COMPONENTS + 1))  # Counts for the summary sweep.
WEIGHT_N_COMPONENTS = 2  # First N fitted components; must be <= N_COMPONENTS.
WEIGHT_N_CLUSTERS = 'auto'  # Use an integer for a fixed count.
WEIGHT_K_MIN = 2
WEIGHT_K_MAX = 10
WEIGHT_USE_THRESHOLDED = False
WEIGHT_METHODS = ['kmeans', 'minibatch', 'gmm', 'birch']
WEIGHT_METHOD = 'kmeans'  # Choose a computed method for plotting.
WEIGHT_CLUSTER_INPUT = OUTPUT_DIR

if base is not None:
    export_inputs(weights, coverage_region_metadata, WEIGHT_CLUSTER_INPUT)
    for dimensions in WEIGHT_COMPONENT_COUNTS:
        command = ['python', '-u', str(ROOT / 'vs3_1_weight_clustering.py'),
            '--input-dir', str(WEIGHT_CLUSTER_INPUT), '--n-clusters', str(WEIGHT_N_CLUSTERS),
            '--n-components', str(dimensions), '--k-min', str(WEIGHT_K_MIN), '--k-max', str(WEIGHT_K_MAX),
            '--seed', str(SEED), '--methods', *WEIGHT_METHODS]
        if WEIGHT_USE_THRESHOLDED:
            command.append('--threshold')
        print('# Fit weight clusters:', shlex.join(command), sep='\n')


### Clustering summary across algorithms and component counts
Run the component-count sweep above, then load the saved automatic-selection results here. Each column represents one modality; rows show the selected cluster count and its silhouette score. Missing runs appear as gaps and are listed below. Silhouette values describe separation in different dimensional spaces and are exploratory, not a test of biological validity. The individual plots below still use `WEIGHT_N_COMPONENTS` and `WEIGHT_METHOD`.


In [ ]:
INPUT_DIR = WEIGHT_CLUSTER_INPUT
from vs3_1_weight_clustering import summarize_clustering
clustering_summary = summarize_clustering(INPUT_DIR, WEIGHT_COMPONENT_COUNTS, WEIGHT_METHODS,
    use_thresholded=WEIGHT_USE_THRESHOLDED) if base is not None else None
if clustering_summary is not None:
    display(clustering_summary)

In [ ]:
METHOD_COLORS = {'kmeans': '#0072B2', 'minibatch': '#E69F00', 'gmm': '#009E73', 'birch': '#CC79A7'}

if clustering_summary is not None and not clustering_summary.empty:
    table = clustering_summary
    title = f'{MODEL} · automatic cluster selection'
    import matplotlib.pyplot as plt
    from matplotlib.ticker import MaxNLocator
    modalities = table.modality.unique()
    methods = table.method.unique()
    fig, axes = plt.subplots(2, len(modalities), figsize=(6 * len(modalities), 8), squeeze=False, sharex=True, sharey='row', layout='constrained')
    for col, modality in enumerate(modalities):
        for index, method in enumerate(methods):
            frame = table[(table.modality == modality) & (table.method == method)].sort_values('n_components')
            for row, metric in enumerate(('n_clusters', 'silhouette')):
                axes[row, col].plot(frame.n_components, frame[metric], 'o', color=METHOD_COLORS[method], linestyle=['-', '--', ':', '-.'][index % 4], label=method)
        axes[0, col].set(title=modality, ylabel='Selected number of clusters')
        axes[0, col].yaxis.set_major_locator(MaxNLocator(integer=True))
        axes[1, col].set(xlabel='Number of components', ylabel='Mean silhouette', ylim=(-1, 1), xticks=sorted(table.n_components.unique()))
        axes[0, col].legend()
    fig.suptitle(title or 'Automatic cluster selection across component counts')
    plt.show()


### Plot saved clusters

Select a computed method and component count. Weight-space colors identify clusters; brain maps show absolute-weight norms using `Reds`. Regional percentages describe sampling composition, not enrichment.


In [ ]:
INPUT_DIR = WEIGHT_CLUSTER_INPUT
WEIGHT_METHOD = 'kmeans'
WEIGHT_N_COMPONENTS = 2
cluster_results = None
if base is not None:
    try:
        cluster_results = load_results(INPUT_DIR, WEIGHT_METHOD, n_clusters=WEIGHT_N_CLUSTERS,
            use_thresholded=WEIGHT_USE_THRESHOLDED, n_components=WEIGHT_N_COMPONENTS)
    except FileNotFoundError:
        print('# Run the corresponding clustering command printed above, then reload.')


In [ ]:
FIGSIZE = (12, 5)
if cluster_results is not None and WEIGHT_N_CLUSTERS == 'auto':
    fig, axes = plt.subplots(1, len(cluster_results), figsize=FIGSIZE, squeeze=False, layout='constrained')
    for ax, (modality, result) in zip(axes.flat, cluster_results.items()):
        selection = result['cluster_selection']
        ax.plot(selection.k, selection.silhouette, 'o-', color=MODALITY_COLORS[modality])
        chosen = selection.loc[selection.selected]
        ax.scatter(chosen.k, chosen.silhouette, s=80, facecolors='none', edgecolors='black', label='Selected')
        ax.set(title=modality, xlabel='Requested cluster count', ylabel='Mean silhouette')
        ax.grid(alpha=.2)
        ax.legend()
    plt.show()


In [ ]:
# In the brain plots we should see the color of the labels not the norm of the weights

if cluster_results is not None:
    for modality, result in cluster_results.items():
        title = f'{MODEL} · {modality} · {WEIGHT_METHOD}'
        import matplotlib.pyplot as plt
        from vs3_1_modality_specificity import ordered_regions
        members = result['memberships']
        dimensions = sum((name.startswith('input_weight_') for name in members.columns))
        display_dimensions = 2 if dimensions <= 2 else 3
        title = f'{title} · clustering uses {dimensions} components'
        if dimensions > 3:
            title += '; display: first three'
        clusters = sorted(members.loc[members.cluster_id >= 0, 'cluster_id'].unique())
        colors = {c: plt.get_cmap('turbo')(i / max(len(clusters) - 1, 1)) for i, c in enumerate(clusters)}
        fig = plt.figure(figsize=(10, 8), layout='constrained')
        ax = fig.add_subplot(projection='3d' if display_dimensions == 3 else None)
        for c in clusters:
            xyz = members.loc[members.cluster_id == c].reindex(columns=[f'input_weight_{i}' for i in range(1, display_dimensions + 1)], fill_value=0).to_numpy()
            ax.scatter(*xyz.T, s=9, alpha=0.4, color=colors[c], label=f'C{c} (n={len(xyz)})', rasterized=True)
            ax.scatter(*xyz.mean(axis=0), s=180, marker='*', color=colors[c], edgecolor='black')
        ax.set(xlabel='|Weight 1|', ylabel='|Weight 2|' if dimensions >= 2 else 'Unused axis (zero)', title=f'{title} · stars: member means')
        if display_dimensions == 3:
            ax.set_zlabel('|Weight 3|')
        ax.legend()
        brain = plt.figure(figsize=(12, 5))
        values = np.linalg.norm(members[[f'input_weight_{i}' for i in range(1, dimensions+1)]].to_numpy(), axis=1)
        plotting.plot_markers(values, members[['x', 'y', 'z']].to_numpy(), figure=brain,
            node_cmap='Reds', node_vmin=0, node_size=3, title=title + ' · weight norm')
        table = result['region_contributions']
        regions = ordered_regions(members.region_shared.unique())
        hist, axes = plt.subplots(len(clusters), 1, figsize=(14, max(4, 3 * len(clusters))), squeeze=False, layout='constrained')
        for ax, c in zip(axes.flat, clusters):
            frame = table[table.cluster_id == c].set_index('region_shared').reindex(regions, fill_value=0)
            ax.bar(regions, frame.percent, color=colors[c])
            ax.set(title=f'Cluster {c}', ylabel='% of cluster sources', ylim=(0, 100))
            ax.tick_params(axis='x', rotation=65, labelsize=8)
        hist.suptitle(f'{title} · regional composition (source counts, not weight magnitude)')
        figures = {f'weights_{display_dimensions}d': fig, 'brain_centroids': brain, 'region_contributions': hist}
        plt.show()
